# Clase 10 — Dashboards interactivos con Dash: callbacks y controles

En la clase anterior construiste dashboards con Dash, pero eran **fotografías caras**: se veían bien, pero nadie podía tocar nada. Hoy los vuelves interactivos con **callbacks**: el mecanismo que conecta "el usuario hizo algo" con "el gráfico se actualiza automáticamente".


## ⚠️ Antes de empezar: Dash no corre nativo en Google Colab

Igual que en la clase anterior, Dash levanta un servidor web propio (Flask) que escucha en un puerto local (`127.0.0.1:8050`). Colab ejecuta tu notebook en una máquina remota a la que no tienes acceso directo por esa vía, así que **el patrón `app.run()` normal no te va a abrir el dashboard dentro de Colab**.

Tienes dos opciones para trabajar con los ejemplos de esta clase:

### Opción A — Archivo `.py` en tu terminal o IDE (recomendada para practicar "de verdad")

1. Copia el código del ejemplo a un archivo `.py` (en este notebook lo generamos con la magia `%%writefile`).
2. Corre ese archivo desde una terminal local (no en Colab): `python nombre_archivo.py`.
3. Abre en tu navegador la URL que aparece en la terminal (normalmente `http://127.0.0.1:8050/`).
4. Cambia los controles y observa cómo el callback actualiza el gráfico en vivo.

### Opción B — `app.run(jupyter_mode='inline')` (para ver el resultado sin salir del notebook)

Desde Dash 2.11, `dash.Dash` sabe mostrar la app **embebida dentro de una celda de salida** del notebook (funciona en Colab y en Jupyter clásico) sin instalar nada adicional. El único cambio frente a la Opción A es la última línea:

```python
app = dash.Dash(__name__)
app.layout = ...      # igual que siempre
# ... callbacks igual que siempre ...

app.run(jupyter_mode='inline')   # <- la diferencia clave frente a app.run()
```

En este notebook te damos **ambas variantes** para el ejemplo principal (callback simple de 1 Input/1 Output), para que elijas la que mejor te funcione. Para los ejemplos siguientes seguimos el mismo patrón.

## 1. La analogía: el callback como un mesero

Imagina un restaurante. Tú, sentado en la mesa, pides un plato — eso es el **Input**: el valor que acabas de introducir o seleccionar (escribes un año, eliges una región, mueves un slider...).

El mesero no cocina en tu mesa: va a la cocina, donde alguien prepara el plato según tu pedido — eso es **la función** que tú escribes. Cuando está listo, el mesero vuelve y te lo trae — eso es el **Output**: lo que se actualiza en la pantalla (un gráfico, un texto, si un control queda deshabilitado...).

Tú nunca le dices al mesero "actualiza el plato" de forma manual. El mesero está siempre atento: en cuanto detecta que pediste algo (cambiaste el Input), automáticamente dispara todo el proceso. Eso es exactamente lo que hace el decorador `@app.callback`.

Sin callbacks, Dash es solo una forma elegante de mostrar una imagen fija. Con callbacks, es una aplicación real.


## 2. La anatomía de un callback

Un callback en Dash tiene siempre esta forma:

```python
@app.callback(Output(component_id='...', component_property='...'),
              Input(component_id='...', component_property='...'))
def mi_funcion(valor_del_input):
    # ... calcular algo con valor_del_input ...
    return resultado
```

Tres piezas que siempre van juntas:

- **`Output`**: le dice a Dash *qué componente* de la página se va a actualizar y *qué propiedad* de ese componente cambia.
- **`Input`**: le dice a Dash *qué componente* vigilar y *qué propiedad* de ese componente, al cambiar, debe disparar la función.
- **La función decorada**: recibe como argumento el valor actual del Input (en el mismo orden en que declaraste los Inputs) y debe `return` el nuevo valor para el Output.

`component_id` y `component_property` identifican un componente sin ambigüedad:

- `component_id`: el identificador único que le pusiste al componente en el layout (`id=...`).
- `component_property`: qué atributo de ese componente lees o escribes. Los más comunes:
  - `value` — el valor actual de un `dcc.Input`, `dcc.Dropdown` o `dcc.RadioItems` (típico para **Inputs**).
  - `figure` — la figura de Plotly que muestra un `dcc.Graph` (típico para **Outputs**).
  - `children` — el contenido interno de un `html.Div` (útil cuando insertas un `dcc.Graph` completo dentro de una división vacía).
  - `disabled` — si un control está deshabilitado o no.

**Regla de oro del orden:** el orden de la lista de `Output` debe coincidir con el orden del `return`; el orden de la lista de `Input` debe coincidir con el orden de los argumentos de la función.


## 3. Ejemplo 1 — Un Input, un Output

El caso más simple: un campo numérico donde el usuario escribe un **año**, y un gráfico de líneas que recalcula el **retraso promedio de llegada por mes** para ese año.

Usamos el dataset `airline_data.csv` (Airline Reporting Carrier On-Time Performance), el mismo que ya conoces de clases anteriores.

Primero, preparamos los datos (esta celda sí corre en Colab sin problema — solo es pandas):


In [ ]:
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px

# Leemos el dataset de aerolíneas (mismo dataset que en clases anteriores)
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

airline_data.head()


### Opción A — Guardar como `.py` y correrlo en una terminal

Ejecuta la siguiente celda: crea un archivo `clase10_ejemplo1_callback_simple.py` con la app completa. Después, desde una terminal (fuera de Colab), corre:

```
python clase10_ejemplo1_callback_simple.py
```

y abre la URL que te indique (normalmente `http://127.0.0.1:8050/`).


In [ ]:
%%writefile clase10_ejemplo1_callback_simple.py
# Import required libraries
import pandas as pd
import plotly.graph_objs as go
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

# Read the airline data into pandas dataframe
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

# Create a dash application
app = dash.Dash(__name__)

# Layout: título, campo de entrada para el año, y un dcc.Graph VACÍO
# (la figura la va a rellenar el callback, no se pasa figure= aquí)
app.layout = html.Div(children=[
    html.H1('Airline Performance Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
              type='number', style={'height': '50px', 'font-size': 35}), ],
              style={'font-size': 40}),
    html.Br(),
    html.Br(),
    html.Div(dcc.Graph(id='line-plot')),
])


# Callback: cuando cambia el "value" de input-year, se dispara get_graph()
# y su resultado se convierte en el "figure" de line-plot.
@app.callback(Output(component_id='line-plot', component_property='figure'),
              Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    # Select data based on the entered year
    df = airline_data[airline_data['Year'] == int(entered_year)]

    # Group the data by Month and compute the average over arrival delay time.
    line_data = df.groupby('Month')['ArrDelay'].mean().reset_index()

    fig = go.Figure(data=go.Scatter(x=line_data['Month'], y=line_data['ArrDelay'],
                                     mode='lines', marker=dict(color='green')))
    fig.update_layout(title='Month vs Average Flight Delay Time',
                       xaxis_title='Month', yaxis_title='ArrDelay')
    return fig


# Run the app
if __name__ == '__main__':
    app.run(debug=True)


### Opción B — `app.run(jupyter_mode='inline')` (verlo dentro del notebook)

Mismo layout y mismo callback, cambiando solo la última línea a `app.run(jupyter_mode='inline')` en vez de `app.run()`.

In [ ]:
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objs as go

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Performance Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
              type='number', style={'height': '50px', 'font-size': 35}), ],
              style={'font-size': 40}),
    html.Br(),
    html.Br(),
    html.Div(dcc.Graph(id='line-plot')),
])


@app.callback(Output(component_id='line-plot', component_property='figure'),
              Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    df = airline_data[airline_data['Year'] == int(entered_year)]
    line_data = df.groupby('Month')['ArrDelay'].mean().reset_index()

    fig = go.Figure(data=go.Scatter(x=line_data['Month'], y=line_data['ArrDelay'],
                                     mode='lines', marker=dict(color='green')))
    fig.update_layout(title='Month vs Average Flight Delay Time',
                       xaxis_title='Month', yaxis_title='ArrDelay')
    return fig


# mode='inline' -> la app aparece embebida como salida de esta celda
app.run(jupyter_mode='inline')


Léelo como una frase: "cuando el `value` del componente `input-year` cambie, llama a `get_graph` con ese valor, y el resultado que devuelva se convierte en el nuevo `figure` del componente `line-plot`". La figura se construye con `plotly.graph_objects` (`go.Figure` / `go.Scatter`), igual de válido que Plotly Express.


## 4. Ejemplo 2 — Un Input, múltiples Outputs

Un callback no está limitado a un solo Output. Podemos usar el mismo campo de año como único Input, pero actualizar **varios** gráficos a la vez: retrasos por transportista, clima, sistema aéreo nacional (NAS), seguridad y aeronave tardía. Para eso, tanto los `Output` como el valor de `return` se agrupan en **listas**, en el mismo orden.


In [ ]:
import plotly.express as px

def compute_info(airline_data, entered_year):
    """Separamos el cálculo pesado (agrupar/promediar) de la construcción de gráficos."""
    df = airline_data[airline_data['Year'] == int(entered_year)]
    avg_car = df.groupby(['Month', 'Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Month', 'Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS = df.groupby(['Month', 'Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec = df.groupby(['Month', 'Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late = df.groupby(['Month', 'Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()
    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late


# Así se declara el callback con 5 Outputs y 1 Input (no lo ejecutamos aquí,
# es el mismo patrón de Opción A / Opción B que ya conoces):
#
# @app.callback([
#                Output(component_id='carrier-plot', component_property='figure'),
#                Output(component_id='weather-plot', component_property='figure'),
#                Output(component_id='nas-plot', component_property='figure'),
#                Output(component_id='security-plot', component_property='figure'),
#                Output(component_id='late-plot', component_property='figure')
#                ],
#                Input(component_id='input-year', component_property='value'))
# def get_graph(entered_year):
#     avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, entered_year)
#
#     carrier_fig = px.line(avg_car, x='Month', y='CarrierDelay', color='Reporting_Airline',
#                            title='Average carrier delay time (minutes) by airline')
#     weather_fig = px.line(avg_weather, x='Month', y='WeatherDelay', color='Reporting_Airline',
#                            title='Average weather delay time (minutes) by airline')
#     nas_fig = px.line(avg_NAS, x='Month', y='NASDelay', color='Reporting_Airline',
#                        title='Average NAS delay time (minutes) by airline')
#     sec_fig = px.line(avg_sec, x='Month', y='SecurityDelay', color='Reporting_Airline',
#                        title='Average security delay time (minutes) by airline')
#     late_fig = px.line(avg_late, x='Month', y='LateAircraftDelay', color='Reporting_Airline',
#                         title='Average late aircraft delay time (minutes) by airline')
#
#     # El ORDEN del return debe coincidir con el orden de los Output de arriba
#     return [carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]

# Probamos igualmente la función auxiliar para confirmar que los datos salen bien:
avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, 2010)
avg_car.head()


La regla es simple: el **orden** de la lista de `Output` tiene que coincidir exactamente con el **orden** de la lista que devuelves en el `return`. Aquí también aparece un patrón útil: separar el cálculo pesado (`compute_info`) de la función del callback, que se queda solo con "armar los gráficos".


## 5. Ejemplo 3 — Dos controles que alimentan un mismo callback

Un callback también puede depender de **varios** controles a la vez. Este es el dashboard de incendios en Australia: un `dcc.RadioItems` para elegir región y un `dcc.Dropdown` para elegir año, ambos alimentando el mismo par de gráficos (un **pie chart** y un **bar chart**).

Dataset: incendios forestales en Australia.


In [ ]:
import pandas as pd

# Leemos el dataset de incendios en Australia
df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')

# Extraemos año y mes a partir de la columna Date
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

df.head()


Fíjate en dos detalles clave del layout de este ejemplo:

- Las divisiones de salida están **vacías** (`html.Div([], id='plot1')`) — no llevan un `dcc.Graph` predefinido. El callback va a insertar el gráfico completo dentro de ellas, por eso el `component_property` del Output es `children` y no `figure`.
- La función del callback recibe **dos** argumentos (`input_region`, `input_year`), en el mismo orden en que se declararon los dos `Input`. Da igual cuál de los dos controles cambie el usuario — el callback se dispara y recibe los valores actuales de ambos.

### Opción A — Guardar como `.py` y correrlo en una terminal


In [ ]:
%%writefile clase10_ejemplo3_region_year.py
# Import required libraries
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Read the wildfire data into pandas dataframe
df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')

# Extract year and month from the date column
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

# Create a dash application
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    # Outer division con dos divisiones internas: controles y gráficos
    html.Div([
        html.Div([
            html.H2('Select Region:', style={'margin-right': '2em'}),
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           "NSW", id='region', inline=True)]),

        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Year.unique(), value=2005, id='year')
        ]),

        # Divisiones vacías: aquí el callback va a insertar los dcc.Graph completos
        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])


@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value')])
def reg_year_display(input_region, input_year):
    # data
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    # Plot one - Monthly Average Estimated Fire Area
    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Monthly Average Estimated Fire Area in year {}".format(input_region, input_year))

    # Plot two - Monthly Average Count of Pixels for Presumed Vegetation Fires
    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in year {}'.format(input_region, input_year))

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]


if __name__ == '__main__':
    app.run(debug=True)


### Opción B — `app.run(jupyter_mode='inline')`

In [ ]:
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    html.Div([
        html.Div([
            html.H2('Select Region:', style={'margin-right': '2em'}),
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           "NSW", id='region', inline=True)]),

        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Year.unique(), value=2005, id='year')
        ]),

        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])


@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value')])
def reg_year_display(input_region, input_year):
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Monthly Average Estimated Fire Area in year {}".format(input_region, input_year))

    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in year {}'.format(input_region, input_year))

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]


app.run(jupyter_mode='inline')


## Puntos clave

- Un callback conecta un `Input` (lo que el usuario cambia) con un `Output` (lo que se actualiza) a través de una función que tú escribes.
- `component_id` identifica el componente; `component_property` identifica qué atributo se lee (`value` normalmente) o se escribe (`figure`, `children`, `disabled`, entre otros).
- El orden importa: el orden de los `Output` en la lista debe coincidir con el orden de los valores en el `return`; el orden de los `Input` debe coincidir con el orden de los argumentos de la función.
- Un callback puede tener varios Outputs (Ejemplo 2) y también varios Inputs a la vez (Ejemplo 3) — ambos patrones simplemente usan listas.
- Si el Output apunta a `figure`, devuelves la figura de Plotly directamente. Si apunta a `children` de una división vacía, devuelves el componente completo (por ejemplo `dcc.Graph(figure=fig)`).
- Dash no corre nativo en Colab: usa un archivo `.py` en terminal/IDE, o `app.run(jupyter_mode='inline')` dentro del notebook (sin instalar nada adicional desde Dash 2.11).

Ahora ve al notebook de práctica (`clase10_practica_dash-interactividad.ipynb`) para aplicar estos patrones tú mismo.